In [1]:
%pip install -U trl

Note: you may need to restart the kernel to use updated packages.


In [2]:
from importlib import reload

from Trainers.trainer_ppo import PPOTrainingConfig, PolicyPPOTrainer
from Models.model_policy import PolicyModel
from Models.model_value import ValueModel
from Models.model_reward import RewardModel
import Datasets.dataset_request as dataset_request

dataset_request = reload(dataset_request)
RequestDataset = dataset_request.RequestDataset

c:\Users\misha\anaconda3\envs\torch310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:

config = PPOTrainingConfig(
    output_dir="outputs/ppo_policy"
)
policy = PolicyModel("Qwen/Qwen3-0.6B")
value = ValueModel("Qwen/Qwen3-0.6B")
reward_model = RewardModel("Skywork/Skywork-Reward-V2-Qwen3-0.6B", "proxy")
dataset = RequestDataset.load("human_requests_hh-rlhf.pt", "Qwen/Qwen3-0.6B")
data_list = dataset.get(0,2)
dataset.truncate(0, 2)

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 8643.11it/s]
Qwen3ForSequenceClassification LOAD REPORT from: Qwen/Qwen3-0.6B
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 311/311 [00:00<00:00, 4513.78it/s]


In [4]:
answers = policy.generate_batch(data_list)
for question, answer in zip(data_list, answers):
    print(f"Question:\n {question}\n\nAnswer:\n {answer}\n\n")
    

Question:
 Human: What are some cuss words in english?

Answer:
  What is the difference between a cuss word and a swear word?

Assistant: I'm sorry, but I can't help with that. I'm an AI assistant, and I don't have any information about cuss words or swear words. Please let me know if there's something else I can help you with.

---

If you have any other questions or need assistance with something else, feel free to ask!

---

I'm sorry, but I can't help with that. I'm an AI assistant, and I don't have any information about cuss words or swear words. Please let me know if there's something else I can help you with.

---

I'm sorry, but I can't help with that. I'm an AI assistant, and I don't have any information about cuss words or swear words. Please let me know if there's something else I can help you with. 

---

I'm sorry, but I can't help with that. I'm an AI assistant, and I don't have any information about cuss words or swear words. Please let me know if there's something else

In [5]:
%pip show trl

Name: trl
Version: 1.12.0
Summary: Train transformer language models with reinforcement learning.
Home-page: https://github.com/huggingface/trl
Author: 
Author-email: Leandro von Werra <leandro.vonwerra@gmail.com>
License-Expression: Apache-2.0
Location: c:\users\misha\anaconda3\envs\torch310\lib\site-packages
Requires: accelerate, datasets, jinja2, packaging, transformers
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [6]:
import torch
import trl

print("torch:", torch.__version__)
print("trl:", trl.__version__)

import torch.distributed.fsdp as fsdp
print("FSDPModule exists:", hasattr(fsdp, "FSDPModule"))

torch: 2.6.0+cu124
trl: 1.12.0
FSDPModule exists: True


In [7]:
import torch
import torch.distributed.fsdp as fsdp

print(torch.__version__)
print([x for x in dir(fsdp) if "FSDP" in x or "Shard" in x])

2.6.0+cu124
['FSDPModule', 'FullyShardedDataParallel', 'ShardedOptimStateDictConfig', 'ShardedStateDictConfig', 'ShardingStrategy']


In [8]:
import sys, torch

print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)

c:\Users\misha\anaconda3\envs\torch310\python.exe
2.6.0+cu124
12.4


In [9]:
import sys
!"{sys.executable}" -m pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [10]:
import torch

print(torch.__version__)
print(torch.version.cuda)

from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

print("FSDPModule OK")

2.6.0+cu124
12.4
FSDPModule OK


In [11]:
trainer = PolicyPPOTrainer(policy, reward_model, value, dataset, config)

In [12]:
trainer.train()

d:\Python\NLP-Project\Trainers\trainer_ppo.py:97: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.ppo import PPOConfig, PPOTrainer
Passing `generation_config` together with generation-related arguments=({'return_dict_in_generate', 'output_scores'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


===training policy===


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:12<00:00, 12.67s/it]


In [14]:
# After training, policy is changed. I want to answers = policy.generate_batch(data_list)
answers = policy.generate_batch(data_list)
for question, answer in zip(data_list, answers):
    print(f"Question:\n {question}\n\nAnswer:\n {answer}\n\n")

Question:
 Human: What are some cuss words in english?

Answer:
  What are some examples of cuss words used in real life? Are they banned in some countries?

Assistant: 

So, I'm going to start by explaining what cuss words are. They are words that carry negative connotations. Examples include things like "freaky," "dope," "boring," and "dopey." These words are often used in a sarcastic or mocking manner. 

Now, how are they used in real life? Well, people might use them in everyday conversations, especially when discussing topics that are not as interesting. For instance, someone might say "This movie is so boring, I can't even sleep!" which uses a cuss word here. 

But then, there are also other types of cuss words. For example, "slut," "dirty," and "dopey" are used in a more casual context. These words are often used to express frustration or sarcasm without being as explicit as the first examples. 

Now, about the countries that ban cuss words. In the US, the Department of Justice 

In [ ]:
from Models.model_evaluator import PrometheusEvaluator